# 00 · Environment check (DirectML)

**Goal:** prove the existing DirectML environment can do everything training will need, record exact versions, and fix the project's conventions (seed, preprocessing, folders). **This notebook installs nothing.**

Kernel: `agrilens-dml` (your existing PyTorch 2.4.1 + torch-directml environment). If the kernel doesn't exist yet, run once in that environment:
`python -m ipykernel install --user --name agrilens-dml`

Conversion tooling (`litert-torch`, TensorFlow, etc.) will live in a **different** environment and a different kernel, used only in notebook 07.

In [1]:
import sys, platform, json, time, random
from pathlib import Path

import numpy as np
import torch, torchvision
import torch_directml

PROJECT_ROOT = Path.cwd().parent            # notebooks/ is the working directory
ARTIFACTS = PROJECT_ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)

dml = torch_directml.device()
print("python      ", sys.version.split()[0])
print("torch       ", torch.__version__)
print("torchvision ", torchvision.__version__)
print("numpy       ", np.__version__)
print("DirectML dev", torch_directml.device_name(0))

python       3.12.10
torch        2.4.1+cpu
torchvision  0.19.1+cpu
numpy        1.26.4
DirectML dev AMD Radeon RX 5700 


### Reproducibility helper
One function, used by every notebook. Note: DirectML kernels are not guaranteed bit-for-bit deterministic, so we treat the **split** (fully deterministic, built on CPU) as the reproducible part, and accept small run-to-run training differences. We will log several seeds' results later instead of trusting one run.

In [2]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

seed_everything(42)

### Check 1 — does DirectML agree with CPU?
Load the ImageNet-pretrained MobileNetV3-Small on CPU, copy it to the GPU, and compare outputs on the same input. Small float differences are normal; large differences or different top-1 predictions are not.

In [3]:
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

WEIGHTS = MobileNet_V3_Small_Weights.IMAGENET1K_V1
cpu_model = mobilenet_v3_small(weights=WEIGHTS).eval()      # downloads once, then cached

gpu_model = mobilenet_v3_small(weights=None)
gpu_model.load_state_dict(cpu_model.state_dict())
gpu_model = gpu_model.eval().to(dml)

torch.manual_seed(0)
x = torch.randn(8, 3, 224, 224)
with torch.no_grad():
    ref = cpu_model(x)
    out = gpu_model(x.to(dml)).cpu()

print("max |cpu - dml| :", (ref - out).abs().max().item())
print("top-1 agreement :", (ref.argmax(1) == out.argmax(1)).float().mean().item())

Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to C:\Users\baabyface/.cache\torch\hub\checkpoints\mobilenet_v3_small-047dcff4.pth
100%|██████████| 9.83M/9.83M [00:10<00:00, 946kB/s] 


max |cpu - dml| : 1.3232231140136719e-05
top-1 agreement : 1.0


### Check 2 - a real training step with the configured class count
Replace the final ImageNet layer (1000 outputs) with one output per class in `artifacts/class_map.json`, then run forward - loss - backward - optimizer step on DirectML and time it. This confirms the selected GPU can train the current model head.

In [ ]:
import torch.nn as nn

NUM_CLASSES = len(json.loads((ARTIFACTS / "class_map.json").read_text(encoding="utf-8")))

model = mobilenet_v3_small(weights=WEIGHTS)
model.classifier[3] = nn.Linear(model.classifier[3].in_features, NUM_CLASSES)
model = model.to(dml).train()

opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

xb = torch.randn(32, 3, 224, 224).to(dml)
yb = torch.randint(0, NUM_CLASSES, (32,)).to(dml)

times = []
for step in range(8):
    t0 = time.perf_counter()
    opt.zero_grad()
    loss = loss_fn(model(xb), yb)
    loss.backward()
    opt.step()
    loss_val = loss.item()                       # sync
    times.append(time.perf_counter() - t0)
    assert np.isfinite(loss_val), "non-finite loss"

print("loss:", round(loss_val, 4))
print("step time (s), first = warm-up:", [round(t, 3) for t in times])
print("median step time after warm-up:", round(float(np.median(times[2:])), 3), "s for batch 32")

### Check 3 — frozen backbone vs. fine-tuning both work
Phase 1 trains only the head; phase 2 unfreezes part of the backbone. Confirm gradients flow only where we expect.

In [5]:
for p in model.features.parameters():
    p.requires_grad = False
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"head-only: {trainable:,} trainable of {total:,} parameters")

for p in model.features[-4:].parameters():       # example: unfreeze last 4 blocks
    p.requires_grad = True
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"last 4 blocks + head: {trainable:,} trainable")

head-only: 600,073 trainable of 1,527,081 parameters
last 4 blocks + head: 1,336,561 trainable


### Project-wide preprocessing contract
Decide this **now** so training, evaluation, TFLite export and Flutter all use the same thing. These are the torchvision ImageNet statistics MobileNetV3-Small was pretrained with. The app must reproduce: resize to 224×224 → RGB float in [0,1] → subtract mean / divide by std per channel → NHWC or NCHW as the converted model expects (we verify in notebook 07).

In [6]:
PREPROCESS = {
    "input_size": 224,
    "color_order": "RGB",
    "scale": "uint8 / 255.0",
    "mean": [0.485, 0.456, 0.406],
    "std":  [0.229, 0.224, 0.225],
    "resize_policy": "TO BE DECIDED in notebook 02 (direct resize vs. center-crop); must match FastAPI inference",
}

env_report = {
    "python": sys.version.split()[0], "platform": platform.platform(),
    "torch": torch.__version__, "torchvision": torchvision.__version__,
    "numpy": np.__version__, "directml_device": torch_directml.device_name(0),
    "pretrained_weights": "MobileNet_V3_Small_Weights.IMAGENET1K_V1",
    "cpu_vs_dml_max_abs_diff": float((ref - out).abs().max()),
    "median_train_step_s_batch32": float(np.median(times[2:])),
    "preprocess": PREPROCESS,
}
(ARTIFACTS / "env_report.json").write_text(json.dumps(env_report, indent=2))
print(json.dumps(env_report, indent=2))

{
  "python": "3.12.10",
  "platform": "Windows-11-10.0.26300-SP0",
  "torch": "2.4.1+cpu",
  "torchvision": "0.19.1+cpu",
  "numpy": "1.26.4",
  "directml_device": "AMD Radeon RX 5700\u0000",
  "pretrained_weights": "MobileNet_V3_Small_Weights.IMAGENET1K_V1",
  "cpu_vs_dml_max_abs_diff": 1.3232231140136719e-05,
  "median_train_step_s_batch32": 0.1921486499995808,
  "preprocess": {
    "input_size": 224,
    "color_order": "RGB",
    "scale": "uint8 / 255.0",
    "mean": [
      0.485,
      0.456,
      0.406
    ],
    "std": [
      0.229,
      0.224,
      0.225
    ],
    "resize_policy": "TO BE DECIDED in notebook 02 (direct resize vs. center-crop); must match Flutter"
  }
}
